# accel-sim silicon anchor — GEMM profiler

Times the 6 GPT-2-block weight GEMMs (forward / backward / Adam) on this GPU.

**Before running:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`.

Writes `silicon_profile.json`, prints it, and auto-downloads it. Bring that file back and run:

```bash
python validate/silicon/compare_silicon.py silicon_profile.json
```


In [ ]:
# ---- config (edit if you want) ---------------------------------------------
# NOTE: on a Colab T4 (Turing) or V100 use DTYPE = "fp16" -- those GPUs have
# NO bf16 tensor cores and bf16 silently falls back to a ~20x-slower kernel.
# bf16 is fine on A100 / L4 / H100.
DTYPE   = "fp16"          # "bf16" | "fp16" | "fp32"
TOKENS  = 8 * 1024        # batch 8 x seq 1024
ITERS   = 50
WARMUP  = 15
OUT     = "silicon_profile.json"
TRACE   = True            # also capture a torch.profiler kernel table


In [ ]:
import json, platform, statistics, sys

D_MODEL, D_FF = 768, 3072
SHAPES = [
    ("q_proj",   D_MODEL, D_MODEL),
    ("k_proj",   D_MODEL, D_MODEL),
    ("v_proj",   D_MODEL, D_MODEL),
    ("attn_out", D_MODEL, D_MODEL),
    ("mlp_up",   D_MODEL, D_FF),
    ("mlp_down", D_FF,    D_MODEL),
]
_DTYPES = {"bf16": "bfloat16", "fp16": "float16", "fp32": "float32"}

import torch
import torch.nn as nn
assert torch.cuda.is_available(), "no CUDA device -- Runtime > Change runtime type > T4 GPU"

device = torch.device("cuda")
dtype = getattr(torch, _DTYPES[DTYPE])
M = TOKENS
gpu = torch.cuda.get_device_name(0)
print(f"GPU: {gpu}   dtype={DTYPE}   tokens={M}   iters={ITERS} (+{WARMUP} warmup)")


In [ ]:
def _bench_layer(K, N):
    lin = nn.Linear(K, N, bias=False).to(device=device, dtype=dtype)
    opt = torch.optim.Adam(lin.parameters(), lr=1e-4)          # fp32 m,v state
    # x requires grad: a real hidden layer's input is upstream, so backward
    # computes BOTH the input gradient (dgrad) and the weight gradient (wgrad).
    x = torch.randn(M, K, device=device, dtype=dtype, requires_grad=True)
    fwd, bwd, optt = [], [], []
    for i in range(WARMUP + ITERS):
        # 5 events, not 4: bracket the loss reduction into "forward" so it
        # isn't silently counted as part of "backward".
        ev = [torch.cuda.Event(enable_timing=True) for _ in range(5)]
        ev[0].record()
        y = lin(x)
        ev[1].record()
        loss = y.float().square().mean()
        ev[2].record()
        loss.backward()
        ev[3].record()
        opt.step()
        opt.zero_grad(set_to_none=True)
        x.grad = None
        ev[4].record()
        torch.cuda.synchronize()
        if i >= WARMUP:
            fwd.append(ev[0].elapsed_time(ev[2]))
            bwd.append(ev[2].elapsed_time(ev[3]))
            optt.append(ev[3].elapsed_time(ev[4]))

    def stat(v):
        return {"mean_ms": statistics.fmean(v),
                "std_ms": statistics.pstdev(v) if len(v) > 1 else 0.0}
    return {"forward": stat(fwd), "backward": stat(bwd), "optimizer": stat(optt)}


def _top_ops():
    from torch.profiler import profile, ProfilerActivity
    lins = [nn.Linear(K, N, bias=False).to(device=device, dtype=dtype)
            for _, K, N in SHAPES]
    xs = [torch.randn(M, K, device=device, dtype=dtype, requires_grad=True)
          for _, K, _ in SHAPES]
    opt = torch.optim.Adam([p for l in lins for p in l.parameters()], lr=1e-4)
    for _ in range(5):
        loss = sum(l(x).float().square().mean() for l, x in zip(lins, xs))
        loss.backward(); opt.step(); opt.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
        loss = sum(l(x).float().square().mean() for l, x in zip(lins, xs))
        loss.backward(); opt.step(); opt.zero_grad(set_to_none=True)
        torch.cuda.synchronize()
    rows = []
    for e in prof.key_averages():
        cuda_us = getattr(e, "self_device_time_total", 0) or \
            getattr(e, "self_cuda_time_total", 0)
        if cuda_us > 0:
            rows.append({"op": e.key, "cuda_ms": cuda_us / 1e3, "count": e.count})
    rows.sort(key=lambda r: -r["cuda_ms"])
    return rows[:15]


In [ ]:
per_layer = {}
agg = {"forward": 0.0, "backward": 0.0, "optimizer": 0.0}
for name, K, N in SHAPES:
    r = _bench_layer(K, N)
    per_layer[name] = {"K": K, "N": N, **r}
    for ph in agg:
        agg[ph] += r[ph]["mean_ms"]
    print(f"  {name:9s}  fwd {r['forward']['mean_ms']:7.3f}  "
          f"bwd {r['backward']['mean_ms']:7.3f}  opt {r['optimizer']['mean_ms']:7.3f} ms")

agg["step"] = sum(agg.values())
print(f"  {'TOTAL':9s}  fwd {agg['forward']:7.3f}  bwd {agg['backward']:7.3f}  "
      f"opt {agg['optimizer']:7.3f}  step {agg['step']:7.3f} ms")


In [ ]:
out = {
    "gpu": gpu, "torch": torch.__version__, "cuda": torch.version.cuda,
    "dtype": _DTYPES[DTYPE], "mac_bytes": 4 if DTYPE == "fp32" else 2,
    "tokens": M, "d_model": D_MODEL, "d_ff": D_FF,
    "iters": ITERS, "warmup": WARMUP,
    "phase_ms": agg, "per_layer": per_layer, "host": platform.platform(),
}
if TRACE:
    try:
        out["top_ops"] = _top_ops()
    except Exception as e:
        print(f"  (kernel trace skipped: {e})")

with open(OUT, "w") as f:
    json.dump(out, f, indent=2)

print("\n===== silicon_profile.json (copy this back if the download fails) =====\n")
print(json.dumps(out, indent=2))

try:
    from google.colab import files
    files.download(OUT)
except Exception as e:
    print(f"\n(auto-download unavailable: {e} -- grab {OUT} from the Files sidebar)")
